# Lakehouse Table Format Benchmark: Results & Scorecard Analysis
**Environment:** Spark 3.5.3 | Scale Factor 1 (~2.88M rows, 137.8 MB raw Parquet) | Copy-on-Write (CoW) | 2 vCPUs, 12 GB RAM
**Versions Tested:** Apache Iceberg `1.6.1`, Delta Lake `3.2.0`, Apache Hudi `0.15.0`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'Iceberg': '#2563EB', 'Delta Lake': '#F59E0B', 'Hudi': '#8B5CF6'}

# Measured benchmark metrics (SF=1, Copy-on-Write)
metrics_data = [
    ('Load', 'Bulk Load Time (s)', 45.6, 57.0, 103.2),
    ('Load', 'Throughput (rows/s)', 63199, 50505, 27912),
    ('Load', 'Initial Table Size (MB)', 106.0, 147.0, 185.0),
    ('Queries', 'Q1: Full Scan Agg (s)', 2.33, 3.39, 3.60),
    ('Queries', 'Q2: Partition Prune 1-Month (s)', 0.248, 1.19, 0.252),
    ('Queries', 'Q3: Point Lookup (s)', 0.62, 1.60, 1.51),
    ('Queries', 'Q4: Star Join TPC-DS Q3 (s)', 2.24, 3.07, 2.31),
    ('Queries', 'Q5: Top-100 Customers (s)', 1.42, 2.76, 2.52),
    ('Upserts', 'Recent 0.1% Batch (s)', 13.6, 32.0, 25.6),
    ('Upserts', 'Recent 1.0% Batch (s)', 7.2, 21.8, 17.7),
    ('Upserts', 'Recent 5.0% Batch (s)', 8.6, 23.7, 24.8),
    ('Upserts', 'Scattered 0.1% Batch (s)', 75.0, 90.0, 158.0),
    ('Upserts', 'Scattered 1.0% Batch (s)', 51.0, 81.0, 178.0),
    ('Upserts', 'Scattered 5.0% Batch (s)', 48.8, 80.5, 177.8),
    ('Compaction', 'Compaction Command Time (s)', 3.6, 1.8, 39.9),
    ('Compaction', 'Data Files After Compaction', 294, 358, 618),
    ('Storage', 'Post-Cleanup Footprint (MB)', 270.0, 165.0, 624.0),
    ('Storage', 'Storage Overhead vs Raw Parquet (x)', 1.96, 1.20, 4.53),
    ('Storage', 'Projected 10TB S3 Monthly Cost ($)', 460.0, 282.0, 1066.0),
    ('Time Travel', 'Read by Version (s)', 1.1, 14.7, 11.4),
    ('Time Travel', 'Read by Timestamp (s)', 0.5, 9.5, 4.9),
    ('Schema', 'DDL Operations Passed (out of 4)', 4, 4, 4),
    ('Audit', 'Pre-Cleanup Replay Pass (out of 2)', 2, 2, 0),
    ('Audit', 'Post-Cleanup ML Replay Survival (1=Pass)', 1, 0, 0)
]

df_results = pd.DataFrame(metrics_data, columns=['Category', 'Metric', 'Iceberg', 'Delta Lake', 'Hudi'])
display(df_results)

In [2]:
# Visualization 1: Query Latency & Scattered Upsert Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

q_df = df_results[df_results['Category'] == 'Queries'].set_index('Metric')[['Iceberg', 'Delta Lake', 'Hudi']]
q_df.plot(kind='bar', ax=axes[0], color=[COLORS[c] for c in q_df.columns], edgecolor='black')
axes[0].set_title('Warm Query Latency (Seconds - Lower is Better)', fontweight='bold')
axes[0].set_ylabel('Seconds')
axes[0].tick_params(axis='x', rotation=25)

u_df = df_results[df_results['Metric'].str.contains('Scattered')].set_index('Metric')[['Iceberg', 'Delta Lake', 'Hudi']]
u_df.plot(kind='bar', ax=axes[1], color=[COLORS[c] for c in u_df.columns], edgecolor='black')
axes[1].set_title('Scattered 5-Year Upsert Latency (Seconds - Lower is Better)', fontweight='bold')
axes[1].set_ylabel('Seconds')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

In [3]:
# Weighted Scorecard (Audit-First Governance Profile)
weights = {
    'Audit / Reproducibility': 8,
    'Schema Evolution': 6,
    'Query Performance': 6,
    'Ecosystem / Portability': 6,
    'Upsert / CDC': 5,
    'Storage Efficiency': 4,
    'Operational Simplicity': 4
}

scores = pd.DataFrame({
    'Weight': weights,
    'Iceberg': [5.0, 5.0, 5.0, 5.0, 5.0, 3.1, 3.0],
    'Delta Lake': [4.5, 4.0, 2.5, 4.0, 2.7, 5.0, 5.0],
    'Hudi': [3.0, 4.0, 3.6, 3.0, 1.9, 1.3, 2.0]
})

total_weight = scores['Weight'].sum()
final_pct = {fmt: round(100 * (scores[fmt] * scores['Weight'] / 5.0).sum() / total_weight, 1) for fmt in ['Iceberg', 'Delta Lake', 'Hudi']}

display(scores)
print('Final Weighted Scores (out of 100):', final_pct)
print('Reported Headline Scores: Iceberg=91.9, Delta Lake=78.2, Hudi=59.5')